# Testing BrainPower AES System
#### The purpose of this notebook is to:
1. Test the ASR and OCR components to justify their inclusion in the AES pipeline to provide
   accessibility and inclusion
3. Test the Transformer component to determine its suitability for providing assistance to human scorers
     

#### The Prompt

In [48]:
prompt = """More and more people use computers, but not everyone agrees that this benefits society. 
    Those who support advances in technology believe that computers have a positive effect on people. 
    They teach hand-eye coordination, give people the ability to learn about faraway places and people, 
    and even allow people to talk online with other people. Others have different ideas. Some experts 
    are concerned that people are spending too much time on their computers and less time exercising, 
    enjoying nature, and interacting with family and friends. 

    Write a letter to your local newspaper in which you state your opinion on the effects computers have 
    on people. Persuade the readers to agree with you."""


#### Rubric Guidelines

In [49]:
import pandas as pd
df = pd.read_csv("./data/rubric guidelines.csv", encoding="utf-8")
df[:]

,score,description,typical element 1,typical element 2,typical element 3,typical element 4
0,1,An undeveloped response that may take a positi...,Contains few or vague details.,Is awkward and fragmented.,May be difficult to read and understand.,May show no awareness of audience.
1,2,An under-developed response that may or may no...,Contains only general reasons with unelaborate...,Shows little or no evidence of organization.,May be awkward and confused or simplistic.,May show little awareness of audience.
2,3,A minimally-developed response that may take a...,Has reasons with minimal elaboration and more ...,Shows some organization.,May be awkward in parts with few transitions.,Shows some awareness of audience.
3,4,A somewhat-developed response that takes a pos...,Has adequately elaborated reasons with a mix o...,Shows satisfactory organization.,May be somewhat fluent with some transitional ...,Shows adequate awareness of audience.
4,5,A developed response that takes a clear positi...,Has moderately well elaborated reasons with mo...,Exhibits generally strong organization.,May be moderately fluent with transitional lan...,May show a consistent awareness of audience.
5,6,A well-developed response that takes a clear a...,Has fully elaborated reasons with specific det...,Exhibits strong organization.,Is fluent and uses sophisticated transitional ...,May show a heightened awareness of audience.


#### Method for Feature Extraction and Accuracy Metrics

In [50]:
import nltk
nltk.download('punkt_tab')
from jiwer import wer, cer, process_words
import re
from pathlib import Path
import numpy as np
import copy

class EssayEditor: #4/aug/2026
    """This class normalizes a body of text and can display the body of text along with 
        its word and sentence counts.
    """
    #REGEX - compile patterns and rules once for efficiency
    REPLACEMENTS = {
        '/': ' or ',
        'slash': 'or',
        '@caps1': 'at caps one',
        '@caps2': 'at caps one',
        '@date1': 'at date one',
        '@organization1': 'at organization one',
        '@organization2': 'at organization two',
        'globe(astronomy)': 'globe astronomy',
    }
    SYMBOL_PATTERN = "|".join(re.escape(key) for key in REPLACEMENTS.keys())
    COMPILED_SYMBOL_PATTERN = re.compile(SYMBOL_PATTERN)
    PUNCTUATION_RULE = re.compile(r"[^\s\w]")#fina all characters that are not words or spaces
    SPACES_RULE = re.compile(r"\s+")#find all occurences of multiple spaces
    
    def __init__(self, text):
        self.text = text
        self.original_text=copy.copy(text)
        self.edit_log = []#stores record of successful exection
        self.is_normalised = False
        sentences = nltk.tokenize.sent_tokenize(self.text)
        sentence_count = len(sentences)
        #Taken prior to normalization which removes sentence punctuation
        self.sentence_count = sentence_count

    def normalise(self):
        self.text = self.text.lower()
        self.edit_log.append("Essay converted to lower case successfully.")
        # Normalise spoken forms and symbol
        self.text = self.COMPILED_SYMBOL_PATTERN.sub(
            lambda match: self.REPLACEMENTS[match.group(0)], 
            self.text
        )
        self.edit_log.append("Symbols and spoken forms normalized successfully.")
        #Clean text
        #Replace all non-words or non-spaces with an empty string
        self.text = self.PUNCTUATION_RULE.sub("", self.text)
        #Replace all spaces with a single space, then remove unwanted spaces
        self.text = self.SPACES_RULE.sub(" ", self.text).strip()        
        self.edit_log.append("Cleaned- punctuation an unwanted spaces removed successfully.")
        self.is_normalised = True

    def get_sentence_count(self):
        return self.sentence_count  
        
    def get_text(self) -> str:#use type hint
        return self.text
        
    def get_word_count(self):
        words= nltk.tokenize.word_tokenize(self.text)
        words = [word for word in words if word.isalnum()]
        word_count= len(words)
        return word_count
        
    def show_normalised_text(self):
        print(self.text)  
        
    def show_original_text(self):
        print(self.original_text)
        
    def show_edit_log(self):
        if self.is_normalised:
            print("\nNormalised:")
            for i, log in enumerate(self.edit_log,1):
                print(f"{i}: {log}")
        else:
            print("\nEssay not normalised")

def get_files_metadata(path_to_folder):
    """This function returns a list of the paths, filenames and extensions for 
        all files in a folder.
        Path must be imported from pathlib
    """
    files_metadata = []
    #Folder with audio files
    folder = Path(path_to_folder)
    for file in folder.iterdir():#Returns Path object
        if file.is_file():
            #Get file path
            path = str(file)
            #Get file exension
            extension = file.suffix
            #Get file name/student's name
            name = file.stem
            #Get file size in bytes convert to mb
            size = round(file.stat().st_size/(1024**2), 2)
            #Add to list
            files_metadata.append({
                "path":path,
                "extension": extension,
                "name":name,
                "size":size
            })
    return files_metadata

def calculate_stats(list_of_nums):
    """This function takes as agrument and used numpy to  calculates, 
        the mean, median, min, max, std_dev of a list. It returns a
        dict. 
    """
    #Convert to numpy array
    vals= np.array(list_of_nums)
    stats = {
        #Calculate mean
        "mean": f"{vals.mean():.2%}",
        #Calculate median
        "median": f"{np.median(vals):.2%}",
        #Calculate min
        "min": f"{vals.min():.2%}",
        #Calculate max
        "max": f"{vals.max():.2%}",
        #Calculate std_dev
        "std_dev": f"{vals.std():.2%}"
    }
    return stats

# Calculate WER and CER -----------adapted from ChapGPT
def calculate_error_rates(reference_essay, component_output):
    word_error_rate = wer(reference_essay, component_output)
    character_error_rate = cer(reference_essay, component_output)
    return {
        "wer": word_error_rate, 
        "cer":character_error_rate
    }

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\cowse\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Unit Testing

In [51]:
# UT001 - Importing the ASAP 1.0 dataset
df = pd.read_csv(
    "./data/asap/training_set_rel3.tsv",
    sep="\t",
    encoding="latin-1"
)
df.head()

,essay_id,essay_set,essay,rater1_domain1,rater2_domain1,rater3_domain1,domain1_score,rater1_domain2,rater2_domain2,domain2_score,...,rater2_trait3,rater2_trait4,rater2_trait5,rater2_trait6,rater3_trait1,rater3_trait2,rater3_trait3,rater3_trait4,rater3_trait5,rater3_trait6
0,1,1,"Dear local newspaper, I think effects computer...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",5,4,NaN,9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",4,3,NaN,7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",5,5,NaN,10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,"Dear @LOCATION1, I know having computers has a...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
#Get structure of dataset
df.columns.tolist()

['essay_id',
 'essay_set',
 'essay',
 'rater1_domain1',
 'rater2_domain1',
 'rater3_domain1',
 'domain1_score',
 'rater1_domain2',
 'rater2_domain2',
 'domain2_score',
 'rater1_trait1',
 'rater1_trait2',
 'rater1_trait3',
 'rater1_trait4',
 'rater1_trait5',
 'rater1_trait6',
 'rater2_trait1',
 'rater2_trait2',
 'rater2_trait3',
 'rater2_trait4',
 'rater2_trait5',
 'rater2_trait6',
 'rater3_trait1',
 'rater3_trait2',
 'rater3_trait3',
 'rater3_trait4',
 'rater3_trait5',
 'rater3_trait6']

In [53]:
#Get Statistics
df["essay_set"].value_counts().sort_index()

essay_set
1    1783
2    1800
3    1726
4    1770
5    1805
6    1800
7    1569
8     723
Name: count, dtype: int64

#### Extracting and Displaying Reference Essay

In [54]:
# UT002 - Extracting the first essay for the first prompt, then displaying it

#Extract essay
prompt1 = df[df["essay_set"] == 1].copy()
first_essay = prompt1.iloc[0]
reference_essay = first_essay["essay"]

#Create EssayEditor object
reference_essay = EssayEditor(reference_essay)

# Display reference essay
reference_essay.show_original_text()


Dear local newspaper, I think effects computers have on people are great learning skills/affects because they give us time to chat with friends/new people, helps us learn about the globe(astronomy) and keeps us out of troble! Thing about! Dont you think so? How would you feel if your teenager is always on the phone with friends! Do you ever time to chat with your friends or buisness partner about things. Well now - there's a new way to chat the computer, theirs plenty of sites on the internet to do so: @ORGANIZATION1, @ORGANIZATION2, @CAPS1, facebook, myspace ect. Just think now while your setting up meeting with your boss on the computer, your teenager is having fun on the phone not rushing to get off cause you want to use it. How did you learn about other countrys/states outside of yours? Well I have by computer/internet, it's a new way to learn about what going on in our time! You might think your child spends a lot of time on the computer, but ask them so question about the economy

#### Reference Essay Feature Extraction

In [55]:
#Display Extracted Features - Word and Sentence Counts (from raw essay)
print(f"Number of words: {reference_essay.get_word_count()}")
print(f"Number of sentences : {reference_essay.get_sentence_count()}")


Number of words: 331
Number of sentences : 16


#### Calculating Character and Word Error Rates

In [56]:
#The essay below has been delebrately modified in a controlled manner to determine if 
#the WER and CER calculations are accurate. The digit '5' has been affixed to the ends
#of each of the first 3 and last 3 words of the essay.

modified_essay = """Dear5 local5 newspaper5, I think effects computers have on people are great learning skills/affects 
    because they give us time to chat with friends/new people, helps us learn about the globe(astronomy) and keeps us 
    out of troble! Thing about! Dont you think so? How would you feel if your teenager is always on the phone with friends! 
    Do you ever time to chat with your friends or buisness partner about things. Well now - there's a new way to chat the 
    computer, theirs plenty of sites on the internet to do so: @ORGANIZATION1, @ORGANIZATION2, @CAPS1, facebook, myspace 
    ect. Just think now while your setting up meeting with your boss on the computer, your teenager is having fun on the 
    phone not rushing to get off cause you want to use it. How did you learn about other countrys/states outside of yours? 
    Well I have by computer/internet, it's a new way to learn about what going on in our time! You might think your child 
    spends a lot of time on the computer, but ask them so question about the economy, sea floor spreading or even about the 
    @DATE1's you'll be surprise at how much he/she knows. Believe it or not the computer is much interesting then in class 
    all day reading out of books. If your child is home on your computer or at a local library, it's better than being out 
    with friends being fresh, or being perpressured to doing something they know isnt right. You might not know where your 
    child is, @CAPS2 forbidde in a hospital bed because of a drive-by. Rather than your child on the computer learning, 
    chatting or just playing games, safe and sound in your home or community place. Now I hope you have reached a point 
    to understand and agree with me, because computers can have great effects on you or child because it gives us time to 
    chat with friends/new people, helps us learn about the globe and believe or not keeps us out of troble. 
    Thank you5 for5 listening5."""

#Calculate Error Rates
error_rates = calculate_error_rates(reference_essay.text, modified_essay)
print(f"Word Error Rate: {error_rates['wer']:.3f} \nCharacter Error Rate:{error_rates['cer']:.3f}")


Word Error Rate: 0.018 
Character Error Rate:0.046


#### Normalized Reference Essay

In [57]:
normalised_essay = EssayEditor(reference_essay.text)
normalised_essay.normalise()
normalised_essay.show_normalised_text()
normalised_essay.show_edit_log()

dear local newspaper i think effects computers have on people are great learning skills or affects because they give us time to chat with friends or new people helps us learn about the globe astronomy and keeps us out of troble thing about dont you think so how would you feel if your teenager is always on the phone with friends do you ever time to chat with your friends or buisness partner about things well now theres a new way to chat the computer theirs plenty of sites on the internet to do so at organization one at organization two at caps one facebook myspace ect just think now while your setting up meeting with your boss on the computer your teenager is having fun on the phone not rushing to get off cause you want to use it how did you learn about other countrys or states outside of yours well i have by computer or internet its a new way to learn about what going on in our time you might think your child spends a lot of time on the computer but ask them so question about the econo

## Component Testing

### Whisper (ASR) - Selecting the best model and configuration through experimentation

In [138]:
import librosa
import os
from faster_whisper import WhisperModel
import time

experiments= [] #stores experiment data as dictinaries 7/08/2026
size_compute_type_pairs= [
    ('small', 'int8_float16'), 
    ('small', 'float16'),      
    ('base', 'int8_float16'), 
    ('base', 'float16'),      
    ('medium', 'int8_float16'), 
    ('medium', 'float16'),  
]

#Get file and metadata
file_metadata= get_files_metadata("./data/audio/test")
if file_metadata:
    print(f"{file_metadata[0]['path']} accessed successfully.")
else:
    print("Error accessing file.")

#Load audio file and set sample rate
audio, rate = librosa.load(
file_metadata[0]["path"], 
sr=16000
)

#Get length of audio
duration = librosa.get_duration(y=audio, sr=rate)
if duration:
    print("Audio successfully loaded.")
else:
    print("Error loading audio.")
    
for j, (m_size, c_type) in enumerate(size_compute_type_pairs, 1):
    
    #Start recording model loading time
    start_time = time.time()
    
    #load a model
    model = WhisperModel(
        model_size_or_path=m_size,
        device="cuda", 
        compute_type=c_type,
        cpu_threads=16,
    )
    #Calculate time taken to load model
    model_load_time = time.time() - start_time

    #Transcribe audio for beam sizes 1 to 5 
    for i in range(1, 6): 
        start_time = time.time()
        segments, info = model.transcribe(
            audio, 
            language="en",
            beam_size=i,
            vad_filter=True 
        )
        #Assemble transcription
        whisper_output = " ".join([segment.text for segment in segments])
        #Calculate transcription time
        inference_time = time.time() - start_time
        #Convert transcription string to EssayEditor Object for normalisation
        whisper_output = EssayEditor(whisper_output)
        #Normalise Essay
        whisper_output.normalise()
        #Calculate Transcription Accuracy
        error_rates = calculate_error_rates(normalised_essay.text, whisper_output.text)
        #ASR - Output Evaluation
        wer_threshold = 0.099
        cer_threshold = 0.059
        #IF both WER and CER below threshold, evaluation passed else evaluation failed
        evaluation_status = "pass" if error_rates['wer'] <= wer_threshold and error_rates['cer'] <= cer_threshold else "fail"
        #Store the details of the experiment as a dict
        details = {
            "Model Size": m_size,
            "Compute Type": c_type,
            "CPU Threads": 16,
            "Beam Size": i,
            "VAD Filter":True,
            "File Size": file_metadata[0]["size"], 
            "File Length":duration,
            "Model Load Time": round(model_load_time, 2),
            "Transcription Time": round(inference_time, 2),
            "Transcription": whisper_output.text,
            "WER": round(error_rates["wer"], 3),
            "CER": round(error_rates["cer"], 3),
            "Evaluation Status": evaluation_status
        }
        #Save experiment details to experiments list
        experiments.append(details)
    print(f"Experiment {j}/6 completed")


data\audio\test\Daniella.wav accessed successfully.
Audio successfully loaded.
Experiment 1/6 completed
Experiment 2/6 completed
Experiment 3/6 completed
Experiment 4/6 completed
Experiment 5/6 completed
Experiment 6/6 completed


#### Display results for configurations

In [139]:
#Display Output as a dataframe
df = pd.DataFrame(experiments)
df= df[["Model Size", 
        "Compute Type", 
        "Beam Size", 
        "Model Load Time", 
        "Transcription Time", 
        "WER", 
        "CER", 
        "Evaluation Status"
       ]]
df.head()


,Model Size,Compute Type,Beam Size,Model Load Time,Transcription Time,WER,CER,Evaluation Status
0,small,int8_float16,1,2.35,5.83,0.100,0.050,fail
1,small,int8_float16,2,2.35,6.31,0.111,0.057,fail
2,small,int8_float16,3,2.35,6.60,0.094,0.046,pass
3,small,int8_float16,4,2.35,6.89,0.092,0.044,pass
4,small,int8_float16,5,2.35,6.65,0.119,0.074,fail


In [140]:
#Experiments 24 and 28 achieved the same transcription accuracy. However they differed greatly in the time taken to load the model. 
#In experiment 24, mixed-precision quantization - int8_float16 was used, along with a beam size of 5. This model took rougly 4 
#seconds to transcribe the audio. The model in experiment 28, with beam size 4 and compute_type set to float16, however, 
#took about 50% less time to load but was more than 3 times slower than the model in experiment 24. It is likely that the reduced 
#precision of int8 which produced smaller numerical values, as compared to float16, reduced the computation load resulting in 
#significantly less transcription time.


#### Display in ascending order of WER and CER accuracy

In [141]:
#Sort dataframe in ascending order
df_sorted = df.sort_values(by=["WER", "CER"], ascending = True)
#Show only columns needed for selecting model specification
df_sorted = df_sorted[["Model Size", "Compute Type", "Beam Size", "Model Load Time", "Transcription Time", "WER", "CER", "Evaluation Status"]]
df_sorted.head()

,Model Size,Compute Type,Beam Size,Model Load Time,Transcription Time,WER,CER,Evaluation Status
24,medium,int8_float16,5,4.10,14.10,0.089,0.038,pass
28,medium,float16,4,2.12,43.81,0.089,0.038,pass
22,medium,int8_float16,3,4.10,13.34,0.092,0.039,pass
29,medium,float16,5,2.12,41.47,0.092,0.039,pass
3,small,int8_float16,4,2.35,6.89,0.092,0.044,pass


### Whisper (ASR) Batch Transcription

In [142]:
#The model configuration chosen from the results of the previous experiments is:
# model_size = medium, beam_size=5, compute_type=int8_float16 
from faster_whisper import WhisperModel
import librosa
import time

#Get a list with all file paths, extensions, names in audio folder
files_metadata = get_files_metadata("./data/audio")

#Start recording model loading time
start_time = time.time()
#Load model
model = WhisperModel(
    model_size_or_path="medium",
    device="cuda", 
    compute_type="int8_float16",
    cpu_threads=16,
)
#Calculate model load time
model_load_time = time.time() - start_time

print(f"Whisper loaded successfully in {round(model_load_time,2)} seconds.")
#10/8/2026 converted to object
batch_transcription= {
    "details": [],
    "wer_stats": {
        "mean":'',
        "median":'',
        "min":'',
        "max":'',
        "std_dev":'',
    },
    "cer_stats": {
        "mean":'',
        "median":'',
        "min":'',
        "max":'',
        "std_dev":'',     
    },
    "eval_stats":{    
        "wer passes": 0,
        "wer fails": 0,        
        "wer pass percentage": 0,        
        "wer fail percentage": 0,        
        "cer passes": 0,
        "cer fails": 0,
        "cer pass percentage":0,
        "cer fail percentage":0,
        "total passes": 0,
        "total fails": 0,
        "total pass percentage": 0,
        "total fail percentage": 0,
    }
} 


#Start Transcription Timer
start_time = time.time()

#Transcribe batch of audio files
for file in files_metadata:
    #Load audio file and set sample rate
    audio, rate = librosa.load(file["path"], sr=16000)
    
    #Get length of audio
    duration = librosa.get_duration(y=audio, sr=rate)
    segments, info = model.transcribe(
        audio, 
        language='en',
        beam_size=5,
        vad_filter=True 
    )
    #Assemble transcription
    whisper_output = " ".join([segment.text for segment in segments])

    #Convert to EssayEditor Object for normalisation
    whisper_output = EssayEditor(whisper_output)
    
    #Normalise Essay
    whisper_output.normalise()
    
    #Calculate Error Rates
    error_rates = calculate_error_rates(normalised_essay.get_text(), whisper_output.get_text())
    
    #ASR - Output Evaluation
    wer_threshold = 0.090
    cer_threshold = 0.050
    #IF both WER and CER below threshold evaluation passed else evaluation failed
    evaluation_status = "pass" if error_rates["wer"] <= wer_threshold and error_rates["cer"] <= cer_threshold else "fail"
    
    #Update the pass and fail counts accordingly
    if evaluation_status == "fail":#Either one or both wer/cer failed
        if error_rates["wer"] > wer_threshold and error_rates["cer"] <= cer_threshold: #wer failed, cer passed
            batch_transcription["eval_stats"]["wer fails"] = batch_transcription["eval_stats"]["wer fails"] + 1 
            batch_transcription["eval_stats"]["cer passes"] = batch_transcription["eval_stats"]["cer passes"] + 1 
        elif error_rates["wer"] <= wer_threshold and error_rates["cer"] > cer_threshold: #wer passed, cer failed
            batch_transcription["eval_stats"]["wer passes"] = batch_transcription["eval_stats"]["wer passes"] + 1 
            batch_transcription["eval_stats"]["cer fails"] = batch_transcription["eval_stats"]["cer fails"] + 1 
        else: #They both failed
            batch_transcription["eval_stats"]["cer fails"] = batch_transcription["eval_stats"]["cer fails"] + 1  
            batch_transcription["eval_stats"]["wer fails"] = batch_transcription["eval_stats"]["wer fails"] + 1  
        #Update total fail count
        batch_transcription["eval_stats"]["total fails"] = batch_transcription["eval_stats"]["total fails"] + 1 
    else: #The both passed
        batch_transcription["eval_stats"]["cer passes"] = batch_transcription["eval_stats"]["cer passes"] + 1  
        batch_transcription["eval_stats"]["wer passes"] = batch_transcription["eval_stats"]["wer passes"] + 1  
        batch_transcription["eval_stats"]["total passes"] = batch_transcription["eval_stats"]["total passes"] + 1 
    
    #Store the details of the transcription as a dict
    details = {
        "Student_ID": file["name"],
        "File Path": file["path"],
        "File Size": round(file['size'], 2), 
        "File Length":round(duration,2),
        "File Extension":file["extension"],
        #decided to store the whole object whisper_output instead of just the transcription, whisper_output.text
        #to all for inspection and analysis of both the original and normalised transcription
        "Transcription": whisper_output, #14/8/2026 passing object instead
        "WER": error_rates["wer"],
        "CER": error_rates["cer"],
        "Evaluation Status": evaluation_status
    }
    #Save transcription details to transcription list
    batch_transcription["details"].append(details)
    
#End of transcription, details saved to list
inference_time = time.time() - start_time
print(f"Whisper successfully transcribed {len(batch_transcription['details'])} files in {round(inference_time/60,2)} minutes.")

#Calcualate Statistics for WER, CER

wer_list, cer_list=[],[]#Stores wer and cer statistics

#Store wer values in a list
for detail_obj in batch_transcription["details"]:
    wer_list.append(detail_obj["WER"])
#Get WER stats for Whisper
wer_stats = calculate_stats(wer_list)

#Store wer statistics in transcriptions object
batch_transcription["wer_stats"]["mean"] = wer_stats["mean"]
batch_transcription["wer_stats"]["median"] = wer_stats["median"]
batch_transcription["wer_stats"]["min"] = wer_stats["min"]
batch_transcription["wer_stats"]["max"] = wer_stats["max"]
batch_transcription["wer_stats"]["std_dev"] = wer_stats["std_dev"]
print("WER statistics calculation complete.")

#Store cer values in a list
for detail_obj in batch_transcription["details"]:
    cer_list.append(detail_obj["CER"])
#Get CER stats for Whisper
cer_stats = calculate_stats(cer_list)

#Store cer statistics in transcriptions object
batch_transcription["cer_stats"]["mean"] = cer_stats["mean"]
batch_transcription["cer_stats"]["median"] = cer_stats["median"]
batch_transcription["cer_stats"]["min"] = cer_stats["min"]
batch_transcription["cer_stats"]["max"] = cer_stats["max"]
batch_transcription["cer_stats"]["std_dev"] = cer_stats["std_dev"]
print("CER statistics calculation complete.")

#Calculating batch pass/fail statistics
num_transcriptions = len(batch_transcription['details'])
total_pass_percentage = f"{batch_transcription['eval_stats']['total passes'] / num_transcriptions: .2%}"
total_fail_percentage = f"{batch_transcription['eval_stats']['total fails'] / num_transcriptions: .2%}"
batch_transcription["eval_stats"]['total pass percentage'] = total_pass_percentage
batch_transcription["eval_stats"]['total fail percentage'] = total_fail_percentage

wer_fail_percentage = f"{batch_transcription['eval_stats']['wer fails'] / num_transcriptions: .2%}"
cer_fail_percentage = f"{batch_transcription['eval_stats']['cer fails'] / num_transcriptions: .2%}"
wer_pass_percentage = f"{batch_transcription['eval_stats']['wer passes'] / num_transcriptions: .2%}"
cer_pass_percentage = f"{batch_transcription['eval_stats']['cer passes'] / num_transcriptions: .2%}"
batch_transcription["eval_stats"]['wer pass percentage'] = wer_pass_percentage
batch_transcription["eval_stats"]['wer fail percentage'] = wer_fail_percentage
batch_transcription["eval_stats"]['cer pass percentage'] = cer_pass_percentage
batch_transcription["eval_stats"]['cer fail percentage'] = cer_fail_percentage
print("Evaluation statistics completed successfully.")

Whisper loaded successfully in 3.7 seconds.
Whisper successfully transcribed 27 files in 5.57 minutes.
WER statistics calculation complete.
CER statistics calculation complete.
Evaluation statistics completed successfully.



#### Batch Transcriptions Technical Details

In [143]:
#Display transcription data
df = pd.DataFrame(batch_transcription["details"], index=range(1, len(batch_transcription["details"])+1))
df= df[[
        "Student_ID",
        "File Path",
        "File Size", 
        "File Length",
        "WER",
        "CER",
        "Evaluation Status"
]]
df

,Student_ID,File Path,File Size,File Length,WER,CER,Evaluation Status
1,Aaliah,data\audio\Aaliah.wav,10.60,115.82,0.080556,0.032551,pass
2,Alliah,data\audio\Alliah.wav,11.98,130.87,0.077778,0.019210,pass
3,Amanda,data\audio\Amanda.wav,26.82,146.45,0.130556,0.057631,fail
4,AnonOne,data\audio\AnonOne.wav,13.48,147.23,0.097222,0.043757,fail
5,AnonThree,data\audio\AnonThree.wav,12.95,141.45,0.069444,0.024013,pass
6,AnonTwo,data\audio\AnonTwo.wav,11.73,139.42,0.105556,0.035752,fail
7,Bevan,data\audio\Bevan.wav,12.69,138.55,0.111111,0.053895,fail
8,Brice,data\audio\Brice.wav,16.48,180.01,0.144444,0.062433,fail
9,Daniella,data\audio\Daniella.wav,16.83,183.77,0.088889,0.037887,pass
10,Darnel,data\audio\Darnel.wav,19.29,210.69,0.180556,0.109392,fail


#### Descriptive Statistics

In [144]:
#Statistical Accuracy Statistics
transcription_statistics = {#10/08/2026
    "WER": batch_transcription["wer_stats"],
    "CER": batch_transcription["cer_stats"]
}
df = pd.DataFrame.from_dict(transcription_statistics)
df

,WER,CER
mean,11.55%,5.47%
median,10.56%,4.70%
min,6.67%,1.92%
max,21.94%,13.13%
std_dev,3.76%,2.82%


In [145]:
# Pass and Fail Statistics
df = pd.DataFrame.from_dict(
    batch_transcription['eval_stats'],orient="index", columns=["Batch Stats"])
df


,Batch Stats
wer passes,8
wer fails,19
wer pass percentage,29.63%
wer fail percentage,70.37%
cer passes,15
cer fails,12
cer pass percentage,55.56%
cer fail percentage,44.44%
total passes,8
total fails,19


#### Threshold Performance 

In [146]:
batch_transcription["details"].sort(key=lambda item: item["WER"])
df = pd.DataFrame(batch_transcription["details"], index=range(1, len(batch_transcription["details"])+1))
df= df[[
        "Student_ID",
        "WER",
        "CER",
        "Evaluation Status"
]]
df

,Student_ID,WER,CER,Evaluation Status
1,Neilage,0.066667,0.024546,pass
2,AnonThree,0.069444,0.024013,pass
3,Krista,0.069444,0.027215,pass
4,Alliah,0.077778,0.019210,pass
5,Aaliah,0.080556,0.032551,pass
6,Tia,0.080556,0.032551,pass
7,Daniella,0.088889,0.037887,pass
8,Skyla,0.088889,0.045891,pass
9,Michia,0.094444,0.046958,fail
10,AnonOne,0.097222,0.043757,fail


#### Representative Transcriptions

In [158]:
#Calculate indices for 5 representative samples from WER distribution
#Select samples in increasing WER from min to max
min_indx = 0
max_indx = len(batch_transcription['details']) -1
median_indx = int((min_indx + max_indx)/2)
lower_mid_indx = int((min_indx + median_indx)/2)
upper_mid_indx = int((median_indx + max_indx)/2)

sample_indx = [min_indx, lower_mid_indx, median_indx, upper_mid_indx, max_indx]

for i in sample_indx:
    print(f"Student ID: {batch_transcription['details'][i]['Student_ID']}")
    print(f"WER: {batch_transcription['details'][i]['WER']:.2%}")
    print(f"CER: {batch_transcription['details'][i]['CER']:.2%}")
    print(f"Evaluation Status: {batch_transcription['details'][i]['Evaluation Status']}")
    batch_transcription['details'][i]["Transcription"].show_normalised_text()
    print('\n')


Student ID: Neilage
WER: 6.67%
CER: 2.45%
Evaluation Status: pass
dear local newspaper i think apex computers have on people are great learning skills or effects because they give us time to chat with friends or new people helps us learn about the globe astronomy and keeps us out of trouble think about dont you think so how would you feel if your teenager is always on the phone with friends do you ever tend to chat with your friends or base this partner about things well now theres a new way to chat the computer theres plenty of sites on the internet to do so at organization one at organization two at caps one facebook myspace etc just think now while youre setting up meeting with your boss on the computer your teenager is having fun on the phone not rushing to get off because you want to use it how did you learn about other countries or dates outside of yours well i have by computer or internet its a new way to learn about what going on in our time you might think your child spends a 

### paddleOCR-OCR Model

In [8]:
from paddleocr import PaddleOCR
#You have to experiment to determine the best configuration like you did for Whisper
ocr = PaddleOCR(
    text_detection_model_name="PP-OCRv6_medium_det",
    text_recognition_model_name="PP-OCRv6_medium_rec",
    engine="transformers",
    use_doc_orientation_classify=True,
    use_doc_unwarping=True,
    use_textline_orientation=True,
    
    text_det_limit_side_len=text_det_limit_side_len,
    text_det_limit_type=text_det_limit_type,
    text_det_thresh=text_det_thresh,
    text_det_box_thresh=text_det_box_thresh,
    text_det_unclip_ratio=text_det_unclip_ratio,
    text_rec_score_thresh=text_rec_score_thresh,
    return_word_box=return_word_box,
    
    text_det_unclip_ratio=1.5,   # default ~1.5; larger box expands region, helps if letters at word edges get cut
)
print("PaddleOCR loaded successfully.")
# return list(
#     224         self.predict_iter(
#     225             input,
#     226             use_doc_orientation_classify=use_doc_orientation_classify,
#     227             use_doc_unwarping=use_doc_unwarping,
#     228             use_textline_orientation=use_textline_orientation,
#     229             text_det_limit_side_len=text_det_limit_side_len,
#     230             text_det_limit_type=text_det_limit_type,
#     231             text_det_thresh=text_det_thresh,
#     232             text_det_box_thresh=text_det_box_thresh,
#     233             text_det_unclip_ratio=text_det_unclip_ratio,
#     234             text_rec_score_thresh=text_rec_score_thresh,
#     235             return_word_box=return_word_box,
#     236         )
#     237     )

C:\Users\cowse\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
C:\Users\cowse\AppData\Local\Programs\Python\Python311\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, 'transformers')
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\cowse\.paddlex\official_models\PP-LCNet_x1_0_doc_ori_safetensors`.


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Creating model: ('UVDoc', None, 'transformers')
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\cowse\.paddlex\official_models\UVDoc_safetensors`.


Loading weights:   0%|          | 0/295 [00:00<?, ?it/s]

Creating model: ('PP-LCNet_x1_0_textline_ori', None, 'transformers')
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\cowse\.paddlex\official_models\PP-LCNet_x1_0_textline_ori_safetensors`.


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Creating model: ('PP-OCRv6_medium_det', None, 'transformers')
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\cowse\.paddlex\official_models\PP-OCRv6_medium_det_safetensors`.


Loading weights:   0%|          | 0/350 [00:00<?, ?it/s]

Creating model: ('PP-OCRv6_medium_rec', None, 'transformers')
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\cowse\.paddlex\official_models\PP-OCRv6_medium_rec_safetensors`.


Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

#### Text Detection, Recognition and Extraction

In [101]:
# Extract text
#1. call get_files_metadata() 
#2. store file paths in a file_paths list
#3. use loop to get predictions
#4. store predictions in predictions list/dict
#5. Create a EssayEditor object for each prediction
#6. Normalise each prediction
#7. Calculate and store metrics for each preciction
#8. Calculate and store metrics for batch

#Get paths for all essays in sample
path_to_folder = "./data/essays"
files_list = get_files_metadata(path_to_folder)
print("Number of essays in sample:", len(path_list))

# Will store predictions for the batch 
batch_extraction= { #TODO: Make into a class
    "details": [],
    "wer_stats": {
        "mean":'',
        "median":'',
        "min":'',
        "max":'',
        "std_dev":'',
    },
    "cer_stats": {
        "mean":'',
        "median":'',
        "min":'',
        "max":'',
        "std_dev":'',     
    },
    "eval_stats":{    
        "wer passes": 0,
        "wer fails": 0,        
        "wer pass percentage": 0,        
        "wer fail percentage": 0,        
        "cer passes": 0,
        "cer fails": 0,
        "cer pass percentage":0,
        "cer fail percentage":0,
        "total passes": 0,
        "total fails": 0,
        "total pass percentage": 0,
        "total fail percentage": 0,
    }
} 
#Loop through files detecting and recognizing - predicting
for file in files_list:
    prediction = ocr.predict(file['path'])
    for result in prediction:
        sentences = result['rec_texts']
        #Combine into a single string
        essay = " ".join(sentence for sentence in sentences)
        #Convert to EssayEditor object
        paddleOCR_extraction = EssayEditor(essay)
        #Normalise extracted text
        paddleOCR_extraction.normalise()
        #Calculate Error Rates
        error_rates = calculate_error_rates(normalised_essay.get_text(), paddleOCR_extraction.get_text())
        #OCR - Output Evaluation
        wer_threshold = 0.090
        cer_threshold = 0.050
        #IF both WER and CER below threshold evaluation passed else evaluation failed
        evaluation_status = "pass" if error_rates["wer"] <= wer_threshold and error_rates["cer"] <= cer_threshold else "fail"
        
        #Update the pass and fail counts accordingly
        if evaluation_status == "fail":#Either one or both wer/cer failed
            if error_rates["wer"] > wer_threshold and error_rates["cer"] <= cer_threshold: #wer failed, cer passed
                batch_extraction["eval_stats"]["wer fails"] = batch_extraction["eval_stats"]["wer fails"] + 1 
                batch_extraction["eval_stats"]["cer passes"] = batch_extraction["eval_stats"]["cer passes"] + 1 
            elif error_rates["wer"] <= wer_threshold and error_rates["cer"] > cer_threshold: #wer passed, cer failed
                batch_extraction["eval_stats"]["wer passes"] = batch_extraction["eval_stats"]["wer passes"] + 1 
                batch_extraction["eval_stats"]["cer fails"] = batch_extraction["eval_stats"]["cer fails"] + 1 
            else: #They both failed
                batch_extraction["eval_stats"]["cer fails"] = batch_extraction["eval_stats"]["cer fails"] + 1  
                batch_extraction["eval_stats"]["wer fails"] = batch_extraction["eval_stats"]["wer fails"] + 1  
            #Update total fail count
            batch_extraction["eval_stats"]["total fails"] = batch_extraction["eval_stats"]["total fails"] + 1 
        else: #The both passed
            batch_extraction["eval_stats"]["cer passes"] = batch_extraction["eval_stats"]["cer passes"] + 1  
            batch_extraction["eval_stats"]["wer passes"] = batch_extraction["eval_stats"]["wer passes"] + 1  
            batch_extraction["eval_stats"]["total passes"] = batch_extraction["eval_stats"]["total passes"] + 1 

        confidence_list = []

        for (text, score) in zip(result["rec_texts"], result["rec_scores"]):
            confidence_list.append(f"{score:.3f} | {text}")
        #Store the details of the transcription as a dict
        details = {
            "Student_ID": file["name"],
            "File Path": file["path"],
            "File Size": round(file['size'], 2), 
            "File Extension":file["extension"],
            "Extraction": paddleOCR_extraction, 
            "WER": error_rates["wer"],
            "CER": error_rates["cer"],
            "Confidence Scores": confidence_list,
            "Evaluation Status": evaluation_status
        }
        # Store in batch extractions
        batch_extraction['details'].append(details)


Task was destroyed but it is pending!
task: <Task pending name='Task-599' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\cowse\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-600' coro=<Kernel.shell_main() running at C:\Users\cowse\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\cowse\AppData\Local\Programs\Python\Python311\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Users\cowse\AppData\Local\Programs\Python\Python311\Lib\tokenize.py:529: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  pseudomatch = _compile(PseudoToken).match(line, pos)
Task was destroyed but it is pending!
task: <Task pending name='Task-600' coro=<Kernel.shell_main() running at C:\Users\cowse\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel\kernelbase.p

Number of essays in sample: 27


In [100]:
#Calcualate Batch Statistics for WER, CER

#Stores wer and cer statistics
wer_list, cer_list=[],[]

#Store wer values in a list
for detail_obj in batch_extraction["details"]:
    wer_list.append(detail_obj["WER"])
    cer_list.append(detail_obj["CER"])
    
#Get WER stats for PaddleOCR
wer_stats = calculate_stats(wer_list)

#Store wer statistics in transcriptions object
batch_extraction["wer_stats"]["mean"] = wer_stats["mean"]
batch_extraction["wer_stats"]["median"] = wer_stats["median"]
batch_extraction["wer_stats"]["min"] = wer_stats["min"]
batch_extraction["wer_stats"]["max"] = wer_stats["max"]
batch_extraction["wer_stats"]["std_dev"] = wer_stats["std_dev"]
print("WER statistics calculation complete.")

#Get CER stats for PaddleOCR
cer_stats = calculate_stats(cer_list)

#Store cer statistics in transcriptions object
batch_extraction["cer_stats"]["mean"] = cer_stats["mean"]
batch_extraction["cer_stats"]["median"] = cer_stats["median"]
batch_extraction["cer_stats"]["min"] = cer_stats["min"]
batch_extraction["cer_stats"]["max"] = cer_stats["max"]
batch_extraction["cer_stats"]["std_dev"] = cer_stats["std_dev"]
print("CER statistics calculation complete.")

#Calculating batch pass/fail statistics
num_extractions = len(batch_extraction['details'])
total_pass_percentage = f"{batch_extraction['eval_stats']['total passes'] / num_extractions: .2%}"
total_fail_percentage = f"{batch_extraction['eval_stats']['total fails'] / num_extractions: .2%}"
batch_extraction["eval_stats"]['total pass percentage'] = total_pass_percentage
batch_extraction["eval_stats"]['total fail percentage'] = total_fail_percentage

wer_fail_percentage = f"{batch_extraction['eval_stats']['wer fails'] / num_extractions: .2%}"
cer_fail_percentage = f"{batch_extraction['eval_stats']['cer fails'] / num_extractions: .2%}"
wer_pass_percentage = f"{batch_extraction['eval_stats']['wer passes'] / num_extractions: .2%}"
cer_pass_percentage = f"{batch_extraction['eval_stats']['cer passes'] / num_extractions: .2%}"
batch_extraction["eval_stats"]['wer pass percentage'] = wer_pass_percentage
batch_extraction["eval_stats"]['wer fail percentage'] = wer_fail_percentage
batch_extraction["eval_stats"]['cer pass percentage'] = cer_pass_percentage
batch_extraction["eval_stats"]['cer fail percentage'] = cer_fail_percentage

print("Evaluation statistics completed successfully.")

WER statistics calculation complete.
CER statistics calculation complete.
Evaluation statistics completed successfully.


#### Threshold Performance

In [106]:
#Display transcription data
batch_extraction["details"].sort(key=lambda item: item["WER"])
df = pd.DataFrame(batch_extraction["details"], index=range(1, len(batch_extraction["details"])+1))
df= df[[
        "Student_ID",
        "WER",
        "CER",
        "Evaluation Status"
]]
df

,Student_ID,WER,CER,Evaluation Status
1,Nubia,0.111111,0.040021,fail
2,Skyla,0.172222,0.060832,fail
3,Tamera,0.200000,0.095518,fail
4,Anon 1,0.225000,0.130736,fail
5,Kaylee,0.230556,0.067236,fail
6,Neilage,0.236111,0.079509,fail
7,Shiela,0.238889,0.124333,fail
8,Anon 2,0.250000,0.120598,fail
9,Hannah,0.252778,0.167022,fail
10,Daniella,0.272222,0.075240,fail


#### Descriptive Statistics

In [75]:
#Statistical Accuracy Statistics
transcription_statistics = {#10/08/2026
    "WER": batch_extraction["wer_stats"],
    "CER": batch_extraction["cer_stats"]
}
df = pd.DataFrame.from_dict(transcription_statistics)
df

,WER,CER
mean,47.37%,20.64%
median,46.67%,18.36%
min,24.72%,5.44%
max,90.28%,75.93%
std_dev,11.61%,13.40%


In [102]:
# Pass and Fail Statistics
df = pd.DataFrame.from_dict(
    batch_extraction['eval_stats'],orient="index", columns=["Batch Stats"])
df

,Batch Stats
wer passes,0
wer fails,27
wer pass percentage,0
wer fail percentage,0
cer passes,1
cer fails,26
cer pass percentage,0
cer fail percentage,0
total passes,0
total fails,27


#### Representative Extractions

In [107]:
#Calculate indices for 5 representative samples from WER distribution
#Select samples in increasing WER from min to max

#TODO: Make all of this into a class

min_indx = 0
max_indx = len(batch_extraction['details']) -1
median_indx = int((min_indx + max_indx)/2)
lower_mid_indx = int((min_indx + median_indx)/2)
upper_mid_indx = int((median_indx + max_indx)/2)

sample_indx = [min_indx, lower_mid_indx, median_indx, upper_mid_indx, max_indx]

for i in sample_indx:
    print(f"Student ID: {batch_extraction['details'][i]['Student_ID']}")
    print(f"WER: {batch_extraction['details'][i]['WER']:.2%}")
    print(f"CER: {batch_extraction['details'][i]['CER']:.2%}")
    print(f"Evaluation Status: {batch_extraction['details'][i]['Evaluation Status']}")
    batch_extraction['details'][i]["Extraction"].show_normalised_text() #change Extraction to Output for universality
    print('\n')

Student ID: Nubia
WER: 11.11%
CER: 4.00%
Evaluation Status: fail
dear local newspaper i think effects computers have on people are great learning skills or affects because they give us time to chat with friends or new people helps us learn about the global castronomy and keeps us out of troble thing about dont you think so how would you feel if your teenager is always on the phone with friends do you ever time to chat with your friends or buis partner about things well now theres a new way fo chat the computer theirs plenty of sites on the internet to do so organization1 organization 2 caps1 facebook myspace ect just think now while your setting up meeting with your bass on the computer your teenager is having fun on the phone not rushing to get off cause you want to use it tow did you learn about other countries or states outside of yours well i have by computer or internet its a new way to learn about what going on in eur time you might think your child speaks alot of the time on the

#### Confidence Scores

In [117]:
#Lowest WER and CER scores
print("Student:", batch_extraction["details"][0]["Student_ID"])
batch_extraction["details"][0]["Confidence Scores"]


Student: Nubia


['0.989 | Dear local newspaper, I think effects computers have on',
 '0.999 | people are great learning skills/affects because they give us',
 '0.998 | time to chat with friends/new people, helps us learn about the',
 '0.998 | global Castronomy) and keeps us out of troble! Thing about!',
 '0.985 | Dont you think so? How would you feel if your teenager is',
 '0.996 | always on the phone with friends! Do you ever time to chat',
 '0.985 | with your friends or buis partner about things. well now-',
 "0.997 | there's a new way fo chat the computer, theirs plenty of sites",
 '0.960 | on the internet to do So: @ ORGANIZATION1, @ ORGANIZATION 2,',
 '0.965 | @ caps1, facebook, myspace ect.',
 '0.993 | Just think now while your setting up meeting with your bass',
 '0.996 | on the computer, your teenager is having fun on the phone not rushing',
 '0.982 | to get off cause you want to use it. tow did you learn about other',
 '0.988 | countries/states outside of yours? Well I have by computer/intern

In [118]:
#Higest WER and CER scores
print("\nStudent:", batch_extraction["details"][26]["Student_ID"])
batch_extraction["details"][26]["Confidence Scores"]


Student: Shemar


['0.644 | D/',
 '0.388 | HC',
 '0.535 | tDHy ',
 '0.536 | cxthmz',
 '0.677 | X',
 '0.642 | 2oson@@',
 '0.337 | I ',
 '0.534 | H?',
 '0.248 | 4',
 '0.830 | entoret, to a mes wey to learn annt s going on in ons timne',
 '0.717 | You uight think var dhid sad alate of tame an the iter oat ash them 2o queton aront the ceaony',
 "0.666 | flor soredg or awen aut Hu@dak's you'll be aurpried at how much he/she iaw.Beliave tarnot",
 '0.805 | anuter o mch intreoting n in dlars all dey reading ont of broods If your cild sit have a',
 '0.411 | [,eth ',
 '0.533 | eth thy o rihYaght t whear ehioe Cena oh',
 '0.386 | R',
 '0.923 | sound in your hore or communty flace',
 '0.440 | seao eeasena',
 '0.517 | bee i e th t ri peohe thae',
 '0.828 | or not bups o out of froute. Phank yom Cor loolag']